# 02 - Data Quality and Feature Engineering

## Definition
Feature engineering translates raw loan records into risk-relevant signals.

## Theory
Raw variables are often weak predictors in isolation. Ratios, interaction terms, and domain scores capture borrower stress and recoverability.

## Financial + business intuition
Collections teams act on burden, delinquency severity, and collateral strength, not just raw income or loan amount.


## Definition
Feature engineering converts raw borrower and loan records into risk-relevant explanatory signals.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
A borrower with low EMI-to-income and high collateral coverage is usually easier to recover.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [ ]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


In [ ]:
df = LoanDataLoader(DATA_PATH).load()
fe = FeatureEngineer()
enriched = fe.engineer(df)
print(enriched.shape)
display(enriched.head())


## Engineered Features (with business purpose)
- `Loan_to_Income_Ratio`: affordability pressure
- `Debt_Burden_Score`: repayment stress
- `Collateral_Coverage_Ratio`: lender protection buffer
- `Recovery_Efficiency_Ratio`: outstanding value per collection attempt
- `EMI_to_Income_Ratio`: monthly installment stress
- `Risk_Exposure_Score`: expected risk-weighted exposure
- `Missed_Payment_Severity`: delinquency intensity
- `Delinquency_Score`: normalized delay risk
- `Recovery_Difficulty_Index`: combined recoverability friction
- `Collection_Intensity_Score`: engagement intensity
- `Behavioral_Risk_Score`: repayment behavior health


In [ ]:
engineered_cols = [
    "Loan_to_Income_Ratio",
    "Debt_Burden_Score",
    "Collateral_Coverage_Ratio",
    "Recovery_Efficiency_Ratio",
    "EMI_to_Income_Ratio",
    "Risk_Exposure_Score",
    "Missed_Payment_Severity",
    "Delinquency_Score",
    "Recovery_Difficulty_Index",
    "Collection_Intensity_Score",
    "Behavioral_Risk_Score",
]
display(enriched[engineered_cols + ["Recovery_Status"]].describe().T)


In [ ]:
corr = enriched[engineered_cols + ["Recovery_Status"]].corr()[["Recovery_Status"]].sort_values("Recovery_Status")
display(corr)

plt.figure(figsize=(8, 10))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Engineered Feature Correlation with Recovery Status")
plt.show()


## Interpretation
The strongest engineered features should be reused in segmentation, risk scoring, and strategy assignment to keep business logic consistent.
